# Logistic Regression

I should be able to program a quick logistic regression. Let's do it!

In [ ]:
from dataclasses import dataclass

import torch
from torch import Generator, Tensor, nn
from torch.distributions import Bernoulli, Uniform
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

# Make up a dataset

For a clean "baby's first" setup, the most natural generative model is:

$$x \sim \mathcal{N}(\mu, \Sigma), \quad y \mid x \sim \text{Bernoulli}(\sigma(w^\top x + b))$$

where $\sigma(z) = (1 + e^{-z})^{-1}$. You pick true $w$, $b$, generate $x$, compute $\sigma(w^{\top} x + b^)$, then Bernoulli-sample $y$. Logistic regression should recover $w^*$ (up to scale degeneracy if data is linearly separable).

Why this works well pedagogically: the Bayes-optimal decision rule is exactly linear, so there's no model mismatch to confuse things.

Other good setups:

Class-conditional Gaussians (the LDA generative model):
$$x \mid y=k \sim \mathcal{N}(\mu_k, \Sigma), \quad y \sim \text{Bernoulli}(\pi)$$

If $\Sigma$ is shared, the posterior $P(y=1 \mid x)$ is exactly logistic in $x$ — so logistic regression is the correct discriminative model for this generator.

Unequal covariances ($\Sigma_0 \neq \Sigma_1$): the true boundary becomes quadratic, so logistic regression will be misspecified. Good for showing model limitations.

Uniform or Laplace features: work fine, just less interpretable geometry.

Recommended starting point for 2D visualization:

In [ ]:
@dataclass
class SampleGenerator(Dataset):

    # Model we'll be learning
    weights = Tensor([5, 15])
    bias: float = 12.0

    # Input dimensions & range
    min_inputs = Tensor([-10, -10])
    max_inputs = Tensor([10, 10])

    # Info
    num_samples: int = 150_000
    seed: int | None = None

    def __post_init__(self) -> None:
        """Get samples, output is (X, y)."""
        if self.seed is not None:
            torch.manual_seed(self.seed)
        input_sampler = Uniform(self.min_inputs, self.max_inputs)
        inputs = input_sampler.sample([self.num_samples])
        outputs = self.weights @ inputs.T + self.bias
        output_sampler = Bernoulli(logits=outputs)
        self._inputs = inputs
        self._outputs = output_sampler.sample()

    def __getitem__(self, key: int | slice) -> tuple[Tensor, Tensor]:
        return self._inputs[key], self._outputs[key]

    def __len__(self) -> int:
        return self._outputs.shape[0]


sample_data = SampleGenerator()
all_input, all_output = sample_data[:]

In [ ]:
split_seed = 123

train_data, validation_data = random_split(
    sample_data, [0.8, 0.2], generator=Generator().manual_seed(split_seed)
)
train_dataloader = DataLoader(train_data, batch_size=10_000)
valid_dataloader = DataLoader(validation_data, batch_size=10_000)

# Model

In [ ]:
def logistic(x: Tensor) -> Tensor:
    return 1.0 / (1.0 + torch.exp(-x))


class LogisticRegression(nn.Module):

    def __init__(
        self, num_weights: int = 2, init_bias: float = 0.0, output_logit: bool = True
    ):
        super().__init__()
        # NOTE params should be randomized
        self.weights = nn.Parameter(Tensor([0.0] * num_weights))
        self.bias = nn.Parameter(Tensor([init_bias]))
        self.output_logit = output_logit

    def forward(self, inputs: Tensor) -> Tensor:
        logits = self.weights @ inputs.T + self.bias
        if self.output_logit:
            return logits
        return logistic(logits)


model = LogisticRegression(init_bias=train_data[:][1].mean()).to(device)
model(all_input.to(device))

# Loss

Cross Entropy Loss

$$l_k = p_k^{y_k}  (1-p_k)^{1-y_k}$$
$$l_k' = \ln(...)$$
$$l_k' = y_k \ln(p_k) + (1-y_k)\ln(1-p_k)$$

In [ ]:
def loss_func(output: Tensor, estimate: Tensor) -> torch.Tensor:
    """Cross entropy loss, pbut I'm writing it myself for education."""
    return -torch.sum(
        output * torch.log(estimate) + (1.0 - output) * torch.log(1.0 - estimate)
    )


loss_func(Tensor([1.0]), Tensor([0.99999]))

In [ ]:
loss_func = nn.BCEWithLogitsLoss()
loss_func

# Optimizer

In [ ]:
learning_rate = 1e-2
optim = AdamW(model.parameters(), lr=learning_rate)
optim

# Training Loop

In [ ]:
epochs = 300
progress_bar = tqdm(range(epochs))
latest_train_loss = -1.0
latest_validation_loss = -1.0

for epoch in progress_bar:
    progress_bar.set_description(
        f"E: {epoch} | Train: {latest_train_loss:.4f} | Val: {latest_validation_loss:.4f}"
    )
    model.train()
    for x, y in train_dataloader:
        x, y = x.to(device), y.to(device)
        model.zero_grad()
        loss = loss_func(model(x), y)
        loss.backward()
        optim.step()
        latest_train_loss = loss.item()

    progress_bar.set_description(
        f"E: {epoch} | Train: {latest_train_loss:.4f} | Val: {latest_validation_loss:.4f}"
    )
    model.eval()
    with torch.inference_mode():
        latest_validation_loss = 0
        for x, y in valid_dataloader:
            x, y = x.to(device), y.to(device)
            latest_validation_loss += loss_func(model(x), y).item()

In [ ]:
# Model we'll be learning
true_weights = Tensor([5, 15])
true_bias: float = 12.0


print(true_weights / true_weights.norm())
print(true_bias / true_weights.norm())

In [ ]:
state_dict = model.state_dict()
print(state_dict["weights"] / state_dict["weights"].norm())
print(state_dict["bias"] / state_dict["weights"].norm())